In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.optim as optim

# Kaggle GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

import logging
logging.getLogger("shap").setLevel(logging.ERROR)

# Keep warning handling local and non-invasive. Monkeypatching sklearn internals
# can hide real failures and may break across sklearn versions.
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"sklearn\.utils\.parallel"
)

# =============================================================================
# STEP 1 — SET YOUR LABEL COLUMN HERE
#
#    label_col = "class3"   # Binary: Attack / Normal
#
# The full detector/evaluation pipeline below is binary. class1/class2 require
# separate multiclass detection labels, metrics, and Benign/Attack fallback logic.
# =============================================================================

label_col = "class3"

# This full detector/evaluation pipeline is binary: Attack vs Normal.
# class1/class2 need separate multiclass-specific detection and reporting logic.
if label_col != "class3":
    raise ValueError(
        "This script's detector and final metrics assume label_col='class3' "
        "(binary Attack/Normal). Adapt the detection and reporting sections "
        "before using class1 or class2."
    )

BINARY_LABEL_MAPPING = {
    "Attack": 0,
    "Normal": 1,
}

# Set this to a real source/session/device column if the dataset provides one.
# With None, rows are split independently and the script reports row isolation,
# not true source isolation.
source_col = None

# =============================================================================
# 1. LOAD DATASET
# =============================================================================
df = pd.read_csv(
    "/kaggle/input/datasets/venusaireddyalla/x-iiotid-dataset/X-IIoTID dataset.csv",
    low_memory=False
)

# ============================================================
# REMOVE EXACT DUPLICATE ROWS
# ============================================================
print("Original dataset shape:", df.shape)

print("Duplicate rows before removal:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after removal:", df.duplicated().sum())
print("Dataset shape after duplicate removal:", df.shape)


print("=" * 90)
print("DIAGNOSTIC")
print("=" * 90)
print(f"{'#':<4} {'Column Name':<40} {'nunique':>8}   Sample Values")
print("-" * 90)
for i, col in enumerate(df.columns):
    samples = df[col].dropna().unique()[:5].tolist()
    print(f"{i:<4} {col:<40} {df[col].nunique():>8}   {samples}")
print("=" * 90)
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst 3 rows:\n{df.head(3).to_string()}\n")

if label_col not in df.columns:
    raise ValueError(
        f"label_col = '{label_col}' not found.\n"
        f"Available: {df.columns.tolist()}"
    )
print(f"[Manual] label_col = '{label_col}'\n")
print(f"Value counts (raw):\n{df[label_col].value_counts()}\n")

# =============================================================================
# 2. DATA SPLIT BEFORE PREPROCESSING FITTING
# =============================================================================
# IMPORTANT:
# The raw dataset is split into source-disjoint groups BEFORE any preprocessing
# object is fitted. This prevents meta-validation sources from influencing:
#   - numeric/categorical feature-type decisions
#   - median imputation
#   - categorical vocabularies
#   - one-hot columns
#   - high-cardinality frequency maps
#   - MinMaxScaler
#
# The final TEST set is also isolated before any preprocessing fitting.
# =============================================================================

df = df.dropna(subset=[label_col]).reset_index(drop=True)

label_variants = ["class1", "class2", "class3"]
drop_cols = [c for c in label_variants if c in df.columns and c != label_col]
df = df.drop(columns=drop_cols)
print(f"Dropped sibling label columns: {drop_cols}")

raw_labels = df[label_col].astype(str).copy()

# Encode the binary target explicitly. LabelEncoder sorts labels
# alphabetically, which would make this script's Attack/Normal assumptions
# fragile.
raw_labels = raw_labels.str.strip()
unexpected_labels = sorted(
    set(raw_labels.unique()) - set(BINARY_LABEL_MAPPING)
)

if unexpected_labels:
    raise ValueError(
        "Unexpected class3 labels. Expected exactly Attack/Normal labels; "
        f"found unsupported values: {unexpected_labels}"
    )

y_all = raw_labels.map(BINARY_LABEL_MAPPING).to_numpy(dtype=np.int64)
label_mapping = BINARY_LABEL_MAPPING.copy()

print(
    f"\nLabel mapping : "
    f"{label_mapping}"
)

if set(np.unique(y_all)) != {0, 1}:
    raise RuntimeError(
        "class3 must contain both Attack and Normal samples."
    )

feature_cols = [c for c in df.columns if c != label_col]

# -------------------------------------------------------------------------
# SOURCE / ROW IDs
# -------------------------------------------------------------------------
if source_col is not None:
    if source_col not in df.columns:
        raise ValueError(
            f"source_col = '{source_col}' not found.\n"
            f"Available: {df.columns.tolist()}"
        )
    configured_source_ids = df[source_col].astype(str).to_numpy()
    if pd.Series(configured_source_ids).duplicated().any():
        raise ValueError(
            "source_col contains repeated source IDs. This script currently "
            "splits rows; use a group-aware split before claiming true "
            "source-level isolation."
        )
    source_ids = np.arange(len(df))
    split_unit_name = "source"
else:
    source_ids = np.arange(len(df))
    split_unit_name = "row"
    print(
        "WARNING: source_col is None. Splits are row-isolated, not true "
        "source/session/device-isolated."
    )


def safe_train_test_split(ids, labels, *, test_size, random_state, name):
    """Use stratification only when sklearn can make a valid split."""
    labels = np.asarray(labels)
    unique, counts = np.unique(labels, return_counts=True)
    n_classes = len(unique)
    n_samples = len(labels)

    if isinstance(test_size, float):
        n_test = int(np.ceil(n_samples * test_size))
    else:
        n_test = int(test_size)

    n_train = n_samples - n_test
    can_stratify = (
        n_classes >= 2
        and counts.min() >= 2
        and n_test >= n_classes
        and n_train >= n_classes
    )

    if can_stratify:
        return train_test_split(
            ids,
            test_size=test_size,
            random_state=random_state,
            stratify=labels
        )

    print(
        f"WARNING: {name} cannot be safely stratified "
        f"(samples={n_samples}, classes={n_classes}, "
        f"min_class_count={counts.min() if len(counts) else 0}, "
        f"test_size={test_size}); using non-stratified split."
    )
    return train_test_split(
        ids,
        test_size=test_size,
        random_state=random_state
    )


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def require_no_overlap(left_ids, right_ids, message):
    overlap = set(left_ids) & set(right_ids)
    if overlap:
        raise RuntimeError(
            f"{message}: found {len(overlap)} overlapping {split_unit_name}(s)."
        )

train_source_ids, test_source_ids = safe_train_test_split(
    source_ids,
    y_all,
    test_size=0.20,
    random_state=42,
    name="development/final-test split"
)

# -------------------------------------------------------------------------
# Reserve an EXACT number of meta-source samples BEFORE fitting ANY
# preprocessing.
#
# n_meta = total ORIGINAL sources reserved for the meta-classifier.
# These sources are completely excluded from base training/validation.
#
# Meta pool:
#   80% -> meta training
#   20% -> meta validation
#
# The remaining development sources are used for:
#   90% -> base training
#   10% -> base validation
# -------------------------------------------------------------------------
n_meta = min(5000, max(10, int(0.10 * len(train_source_ids))))

if n_meta >= len(train_source_ids):
    raise ValueError(
        "n_meta must be smaller than the number of development/training sources."
    )

# First reserve exactly n_meta sources.
base_pool_ids, meta_pool_ids = safe_train_test_split(
    train_source_ids,
    y_all[train_source_ids],
    test_size=n_meta,
    random_state=42,
    name="base/meta split"
)

base_train_ids, base_val_ids = safe_train_test_split(
    base_pool_ids,
    y_all[base_pool_ids],
    test_size=0.10,
    random_state=42,
    name="base-train/base-validation split"
)

meta_train_ids, meta_val_ids = safe_train_test_split(
    meta_pool_ids,
    y_all[meta_pool_ids],
    test_size=0.20,
    random_state=42,
    name="meta-train/meta-validation split"
)

print(f"\nConfigured n_meta (original source samples): {n_meta}")
print(f"  Meta training {split_unit_name}s   : {len(meta_train_ids)}")
print(f"  Meta validation {split_unit_name}s : {len(meta_val_ids)}")

# Explicit raw-source disjointness checks.
require_no_overlap(base_train_ids, base_val_ids, "Base train/base validation overlap")
require_no_overlap(base_train_ids, meta_train_ids, "Base train/meta train overlap")
require_no_overlap(base_train_ids, meta_val_ids, "Base train/meta validation overlap")
require_no_overlap(base_val_ids, meta_train_ids, "Base validation/meta train overlap")
require_no_overlap(base_val_ids, meta_val_ids, "Base validation/meta validation overlap")
require_no_overlap(meta_train_ids, meta_val_ids, "Meta train/meta validation overlap")
require_no_overlap(test_source_ids, train_source_ids, "Development/final-test overlap")

print(f"\n[{split_unit_name.capitalize()}-level split]")
print(f"  Base training {split_unit_name}s     : {len(base_train_ids)}")
print(f"  Base validation {split_unit_name}s   : {len(base_val_ids)}")
print(f"  Meta training {split_unit_name}s     : {len(meta_train_ids)}")
print(f"  Meta validation {split_unit_name}s   : {len(meta_val_ids)}")
print(f"  Final test {split_unit_name}s        : {len(test_source_ids)}")
print(f"  Raw {split_unit_name} overlap checks : PASSED")

# -------------------------------------------------------------------------
# RAW GROUPS
# -------------------------------------------------------------------------
df_features = df[feature_cols].copy()

df_base_train = df_features.iloc[base_train_ids].copy()
df_base_val   = df_features.iloc[base_val_ids].copy()
df_meta_train = df_features.iloc[meta_train_ids].copy()
df_meta_val   = df_features.iloc[meta_val_ids].copy()
df_test_raw   = df_features.iloc[test_source_ids].copy()

y_base_train = y_all[base_train_ids].copy()
y_base_val   = y_all[base_val_ids].copy()
y_meta_train = y_all[meta_train_ids].copy()
y_meta_val   = y_all[meta_val_ids].copy()
y_test       = y_all[test_source_ids].copy()

# -------------------------------------------------------------------------
# PREPROCESSING FIT — BASE TRAINING SPLIT ONLY
# -------------------------------------------------------------------------
numeric_like_cols = []
string_like_cols = []

for col in feature_cols:
    converted = pd.to_numeric(
        df_base_train[col],
        errors="coerce"
    )

    if converted.notna().sum() > 0.5 * len(df_base_train):
        numeric_like_cols.append(col)
    else:
        string_like_cols.append(col)

DUMMIES_THRESHOLD = 20

low_cardinality = [
    c for c in string_like_cols
    if df_base_train[c].nunique(dropna=False) <= DUMMIES_THRESHOLD
]

high_cardinality = [
    c for c in string_like_cols
    if df_base_train[c].nunique(dropna=False) > DUMMIES_THRESHOLD
]

print(f"\nNumeric-like features: {len(numeric_like_cols)}")
print(f"String-like features: {len(string_like_cols)}")
print(f"Low-cardinality categorical: {len(low_cardinality)}")
print(f"High-cardinality categorical: {len(high_cardinality)}")

# -------------------------------------------------------------------------
# Helper: apply preprocessing learned from BASE TRAIN ONLY.
# -------------------------------------------------------------------------
feature_label_encoders = {}
feature_frequency_maps = {}
numeric_medians = {}

# ---- numeric conversion + TRAIN-FITTED medians ----
for col in numeric_like_cols:

    df_base_train[col] = pd.to_numeric(
        df_base_train[col],
        errors="coerce"
    )

    median = df_base_train[col].median()

    if pd.isna(median):
        median = 0.0

    numeric_medians[col] = median

    for frame in [
        df_base_train,
        df_base_val,
        df_meta_train,
        df_meta_val,
        df_test_raw
    ]:
        frame[col] = pd.to_numeric(
            frame[col],
            errors="coerce"
        ).fillna(median)

# ---- categorical missing values ----
for col in string_like_cols:

    for frame in [
        df_base_train,
        df_base_val,
        df_meta_train,
        df_meta_val,
        df_test_raw
    ]:
        frame[col] = (
            frame[col]
            .where(frame[col].notna(), "missing")
            .astype(str)
            .replace({
                "nan": "missing",
                "NaN": "missing",
                "None": "missing",
                "<NA>": "missing",
            })
        )

# ---- high-cardinality encoders fitted ONLY on base training ----
for col in high_cardinality:

    freq_map = (
        df_base_train[col]
        .value_counts(normalize=True)
        .to_dict()
    )

    for frame in [
        df_base_train,
        df_base_val,
        df_meta_train,
        df_meta_val,
        df_test_raw
    ]:
        frame[col] = (
            frame[col]
            .map(freq_map)
            .fillna(0.0)
            .astype(np.float32)
        )

    feature_frequency_maps[col] = freq_map

# ---- one-hot encoder FITTED ONLY on BASE TRAINING ----
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = None

if low_cardinality:
    try:
        onehot_encoder = OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False,
            dtype=np.float32
        )
    except TypeError:
        onehot_encoder = OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse=False,
            dtype=np.float32
        )

    onehot_encoder.fit(
        df_base_train[low_cardinality].astype(str)
    )

    def apply_onehot(frame):
        encoded = onehot_encoder.transform(
            frame[low_cardinality].astype(str)
        )
        encoded_cols = onehot_encoder.get_feature_names_out(
            low_cardinality
        )
        encoded_df = pd.DataFrame(
            encoded,
            index=frame.index,
            columns=encoded_cols
        )
        remaining = frame.drop(
            columns=low_cardinality
        ).copy()
        return pd.concat(
            [remaining, encoded_df],
            axis=1
        )

    df_base_train = apply_onehot(df_base_train)
    df_base_val = apply_onehot(df_base_val)
    df_meta_train = apply_onehot(df_meta_train)
    df_meta_val = apply_onehot(df_meta_val)
    df_test_raw = apply_onehot(df_test_raw)

else:
    print("  No low-cardinality categorical features require one-hot encoding.")

# Align all non-training groups to BASE-TRAIN columns.
base_feature_columns = df_base_train.columns.tolist()

df_base_val = df_base_val.reindex(
    columns=base_feature_columns,
    fill_value=0
)
df_meta_train = df_meta_train.reindex(
    columns=base_feature_columns,
    fill_value=0
)
df_meta_val = df_meta_val.reindex(
    columns=base_feature_columns,
    fill_value=0
)
df_test_raw = df_test_raw.reindex(
    columns=base_feature_columns,
    fill_value=0
)

# ---- final numeric conversion ----
for col in df_base_train.columns:

    for frame in [
        df_base_train,
        df_base_val,
        df_meta_train,
        df_meta_val,
        df_test_raw
    ]:
        frame[col] = pd.to_numeric(
            frame[col],
            errors="coerce"
        ).fillna(0)

# -------------------------------------------------------------------------
# Convert to numpy.
# -------------------------------------------------------------------------
X_base_train_raw = (
    df_base_train.values.astype(np.float32)
)

X_base_val_raw = (
    df_base_val.values.astype(np.float32)
)

X_meta_train_raw = (
    df_meta_train.values.astype(np.float32)
)

X_meta_val_raw = (
    df_meta_val.values.astype(np.float32)
)

X_test_raw = (
    df_test_raw.values.astype(np.float32)
)

# -------------------------------------------------------------------------
# IMPORTANT: MinMaxScaler is FIT ONLY on BASE TRAINING.
# -------------------------------------------------------------------------
scaler = MinMaxScaler()

X_base_train = scaler.fit_transform(
    X_base_train_raw
).astype(np.float32)

X_base_val = scaler.transform(
    X_base_val_raw
).astype(np.float32)

X_meta_train = scaler.transform(
    X_meta_train_raw
).astype(np.float32)

X_meta_val = scaler.transform(
    X_meta_val_raw
).astype(np.float32)

X_test = scaler.transform(
    X_test_raw
).astype(np.float32)

feature_names = df_base_train.columns.tolist()

print("\nFinal feature names:")
for i, f in enumerate(feature_names):
    print(f"{i+1:2d}. {f}")

print("\nTotal features:", len(feature_names))

# Canonical tensors.
X_base_train_t = torch.tensor(
    X_base_train,
    dtype=torch.float32,
    device=device
)

y_base_train_t = torch.tensor(
    y_base_train,
    dtype=torch.long,
    device=device
)

X_base_val_t = torch.tensor(
    X_base_val,
    dtype=torch.float32,
    device=device
)

y_base_val_t = torch.tensor(
    y_base_val,
    dtype=torch.long,
    device=device
)

X_meta_train_t = torch.tensor(
    X_meta_train,
    dtype=torch.float32,
    device=device
)

y_meta_train_t = torch.tensor(
    y_meta_train,
    dtype=torch.long,
    device=device
)

X_meta_val_t = torch.tensor(
    X_meta_val,
    dtype=torch.float32,
    device=device
)

y_meta_val_t = torch.tensor(
    y_meta_val,
    dtype=torch.long,
    device=device
)

X_test_t = torch.tensor(
    X_test,
    dtype=torch.float32,
    device=device
)

y_test_t = torch.tensor(
    y_test,
    dtype=torch.long,
    device=device
)

num_classes = int(len(np.unique(y_all)))
input_dim = X_base_train.shape[1]

print(
    f"\nDataset: {len(df)} samples | "
    f"{input_dim} features | "
    f"{num_classes} classes"
)

print(
    f"Base train: {len(X_base_train)} | "
    f"Base val: {len(X_base_val)} | "
    f"Meta train: {len(X_meta_train)} | "
    f"Meta val: {len(X_meta_val)} | "
    f"Test: {len(X_test)}"
)

# =============================================================================
# STRICT DATA-LEAKAGE CONTROL
#
# SOURCE SPLIT:
#
# Original dataset
#      |
#      +--> Base training sources
#      |
#      +--> Base validation sources
#      |
#      +--> Meta training sources
#      |
#      +--> Meta validation sources
#      |
#      +--> Final test sources
#
# PREPROCESSING:
# All preprocessing FIT operations are performed ONLY on BASE TRAINING:
#   - feature-type decisions
#   - numeric medians
#   - high-cardinality frequency maps
#   - one-hot feature vocabulary
#   - MinMaxScaler
#
# All other groups are TRANSFORMED using those base-training-fitted objects.
#
# MODEL:
#   DNN/RF/SVM -> base training only
#   DNN validation -> base validation only
#   adversarial training -> generated only from base training
#   SHAP background/baseline -> base training only
#   gradient baseline -> base training only
#   variance baseline -> base training only
#   meta-classifier training -> meta-training sources only
#   meta-classifier validation -> meta-validation sources only
#   final evaluation -> final test sources only
# =============================================================================

# =============================================================================
# 5. MODEL
# =============================================================================
class NIDSModel(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = NIDSModel(
    input_dim=input_dim,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()


def train_model(
    model,
    X_t,
    y_t,
    epochs=100,
    lr=1e-3,
    batch_size=1024,
    val_split=0.1,
    patience=10,
    weight_decay=1e-4,
    X_val_t=None,
    y_val_t=None
):
    """
    Train with an explicit validation set when supplied.

    For the leakage-safe experiment, the explicit validation set is always
    constructed from ORIGINAL source IDs that are not in X_t.
    """
    if X_val_t is None or y_val_t is None:
        n_val = max(1, int(len(X_t) * val_split))

        idx = torch.randperm(len(X_t))

        X_trn = X_t[idx[n_val:]]
        y_trn = y_t[idx[n_val:]]

        X_val = X_t[idx[:n_val]]
        y_val = y_t[idx[:n_val]]
    else:
        X_trn = X_t
        y_trn = y_t

        X_val = X_val_t
        y_val = y_val_t

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_trn, y_trn),
        batch_size=batch_size,
        shuffle=True
    )

    criterion_local = nn.CrossEntropyLoss(
        label_smoothing=0.05
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5
    )

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for Xb, yb in loader:

            optimizer.zero_grad()

            loss = criterion_local(
                model(Xb),
                yb
            )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            train_loss += loss.item() * len(Xb)

        train_loss /= len(X_trn)

        model.eval()

        with torch.no_grad():
            val_logits = model(X_val)

            val_loss = criterion_local(
                val_logits,
                y_val
            ).item()

            val_acc = (
                torch.argmax(val_logits, 1) == y_val
            ).float().mean().item()

        scheduler.step(val_loss)

        if (epoch + 1) % 10 == 0:
            print(
                f"  Epoch {epoch+1:>3}/{epochs} | "
                f"train_loss: {train_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_acc: {val_acc:.4f} | "
                f"lr: {optimizer.param_groups[0]['lr']:.2e}"
            )

        if val_loss < best_val_loss - 1e-4:

            best_val_loss = val_loss

            best_state = {
                k: v.clone()
                for k, v in model.state_dict().items()
            }

            no_improve = 0

        else:

            no_improve += 1

            if no_improve >= patience:

                print(
                    f"  Early stopping at epoch {epoch+1} "
                    f"(best val_loss: {best_val_loss:.4f})"
                )

                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    return model


# =============================================================================
# 6. BASE DNN TRAINING
# =============================================================================
print("\n[Phase 2] Training base DNN ...")

model = train_model(
    model,
    X_base_train_t,
    y_base_train_t,
    X_val_t=X_base_val_t,
    y_val_t=y_base_val_t
)

# =============================================================================
# 7. ENSEMBLE MODELS
# =============================================================================
print("\n[Phase 3] Training ensemble classifiers ...")

base_classes, base_class_counts = np.unique(
    y_base_train,
    return_counts=True
)

if len(base_classes) != 2:
    raise RuntimeError(
        "Base training split must contain both Attack and Normal classes."
    )

svm_cv = int(min(3, base_class_counts.min()))

if svm_cv < 2:
    raise RuntimeError(
        "Calibrated SVM requires at least 2 base-training samples in each "
        "class. Increase the dataset size or adjust the split settings."
    )

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_base_train,
    y_base_train
)

svm_model = CalibratedClassifierCV(
    LinearSVC(
        max_iter=2000,
        random_state=42
    ),
    cv=svm_cv
)

svm_model.fit(
    X_base_train,
    y_base_train
)

print(f"  RF and SVM trained using base-training {split_unit_name}s only.")

# =============================================================================
# 6. SHAP UTILITIES
#
# FIX A — get_shap_vector: handles list / 3-D / 2-D outputs without IndexError
# FIX B — safe_shap: always returns exactly n_feats values (trim / pad)
# =============================================================================
def build_explainer(model, background):
    model.eval()
    return shap.DeepExplainer(model, background.clone().detach())


def get_shap_vector(explainer, x, n_feats, label_idx=1):
    """
    Safe 1-D SHAP vector for a single sample x.
    Handles common DeepExplainer layouts:
      - list[class] -> (samples, features)
      - (samples, features, classes)
      - (classes, samples, features)
      - (samples, classes, features)
    """
    shap_vals = explainer.shap_values(x.detach(), check_additivity=False)
    sample_count = x.shape[0]

    if isinstance(shap_vals, list):
        idx = min(label_idx, len(shap_vals) - 1)
        arr = np.array(shap_vals[idx])
        if arr.ndim >= 2 and arr.shape[0] == sample_count:
            arr = arr[0]
        return arr.squeeze()

    arr = np.array(shap_vals)

    if arr.ndim == 3:
        if arr.shape[0] == sample_count and arr.shape[1] == n_feats:
            class_idx = min(label_idx, arr.shape[2] - 1)
            arr = arr[0, :, class_idx]
        elif arr.shape[1] == sample_count and arr.shape[2] == n_feats:
            class_idx = min(label_idx, arr.shape[0] - 1)
            arr = arr[class_idx, 0, :]
        elif arr.shape[0] == sample_count and arr.shape[2] == n_feats:
            class_idx = min(label_idx, arr.shape[1] - 1)
            arr = arr[0, class_idx, :]

    elif arr.ndim == 2 and arr.shape[0] == sample_count:
        arr = arr[0]

    return arr.squeeze()


def safe_shap(explainer, x, n_feats):
    """
    Wrapper that always returns a 1-D array of length exactly n_feats.
    Trims if too long, zero-pads if too short.
    Prevents IndexError when SHAP output length != input_dim.
    """
    vec = np.atleast_1d(
        get_shap_vector(explainer, x, n_feats, label_idx=1)
    ).flatten()
    if len(vec) > n_feats:
        vec = vec[:n_feats]
    elif len(vec) < n_feats:
        vec = np.pad(vec, (0, n_feats - len(vec)))
    return vec

def safe_shap_for_label(explainer, x, n_feats, y):
    """
    Same as safe_shap, but indexes the SHAP output by the GROUND-TRUTH
    label y (Eq. 6: phi_j^(y)), not an arbitrary class.
    """
    label_idx = int(y.item()) if torch.is_tensor(y) else int(y)
    vec = np.atleast_1d(
        get_shap_vector(explainer, x, n_feats, label_idx=label_idx)
    ).flatten()
    if len(vec) > n_feats:
        vec = vec[:n_feats]
    elif len(vec) < n_feats:
        vec = np.pad(vec, (0, n_feats - len(vec)))
    return vec

# IMPORTANT:
# SHAP background/baselines are constructed ONLY from base-training sources.
# Meta-training and meta-validation sources are never used here.
background = X_base_train_t[:min(500, len(X_base_train))].clone().detach()
explainer  = build_explainer(model, background)

model.eval()
n_baseline = min(1000, len(X_base_train))

baseline_shap = np.stack([
    np.abs(
        safe_shap(
            explainer,
            X_base_train_t[i:i+1],
            input_dim
        )
    )
    for i in range(n_baseline)
])
baseline_mean = baseline_shap.mean(axis=0)   # shape: (input_dim,)
devs     = np.linalg.norm(baseline_shap - baseline_mean, axis=1)
LOW_THR  = float(np.percentile(devs, 90))
HIGH_THR = float(np.percentile(devs, 97))
print(f"  Thresholds  Low (p90): {LOW_THR:.4f}  |  High (p97): {HIGH_THR:.4f}")

# =============================================================================
# 7. ATTACKS
# =============================================================================

def shap_pgd_attack(x, y, epsilon=0.1, alpha=0.02, steps=3, top_k=5):
    """
    Faithful implementation of the paper's DeepSHAP-guided masked-PGD.

    Fixes vs. the buggy version:
      1. SHAP computed ONCE on the original x (not re-run on x_adv each step)
      2. SHAP indexed by the ground-truth label y  (Eq. 6: phi^(y)_j)
      3. Top-k mask is FIXED for all K steps (Eqs. 7–8)
    """
    model.eval()
    x_orig  = x.clone().detach()
    x_adv   = x_orig.clone()
    n_feats = x_orig.shape[1]
    # Compute SHAP values ONCE on the ORIGINAL x, indexed by TRUE label y.
    # (bug 1 fix: not re-computed on x_adv; bug 2 fix: uses y, not class 1)
    shap_vec  = safe_shap_for_label(explainer, x_orig, n_feats, y)

    # Select top-k features by absolute SHAP magnitude — FIXED for all steps.
    # (bug 3 fix: mask never changes across iterations)
    important = np.argsort(-np.abs(shap_vec))[:top_k]
    mask      = torch.zeros_like(x_orig)
    mask[0, important] = 1.0
    # K PGD steps, gradient-sign update restricted to top-k features only.
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)

        # Cross-entropy loss w.r.t. ground-truth label y (Eq. 7)
        loss = criterion(model(x_adv), y)
        grad = torch.autograd.grad(
            loss,
            x_adv,
            retain_graph=False,
            create_graph=False
        )[0]

        with torch.no_grad():
            # Update only coordinates in TopFeatures(i, y); others stay at x^(0)
            x_adv = x_adv.detach() + alpha * grad.sign() * mask

            # Project back into ℓ∞ ball of radius ε around original x^(0)
            x_adv = torch.clamp(x_adv, x_orig - epsilon, x_orig + epsilon)

            # Keep values in valid feature domain [0, 1]
            x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()

def deepfool_attack(x,
                    overshoot=0.02,
                    max_iter=20):

    model.eval()

    x_adv = x.clone().detach().requires_grad_(True)

    with torch.no_grad():
        logits = model(x_adv)
        label = logits.argmax(1).item()

    num_classes = logits.shape[1]

    r_tot = torch.zeros_like(x_adv)

    itr = 0

    while itr < max_iter:

        x_adv.requires_grad_(True)

        outputs = model(x_adv)

        current = outputs.argmax(1).item()

        if current != label:
            break

        min_ratio = float("inf")
        best_w = None

        for k in range(num_classes):

            if k == label:
                continue

            model.zero_grad()

            grad_k = torch.autograd.grad(
                outputs[0, k],
                x_adv,
                retain_graph=True
            )[0]

            grad_label = torch.autograd.grad(
                outputs[0, label],
                x_adv,
                retain_graph=True
            )[0]

            w_k = grad_k - grad_label

            f_k = outputs[0, k] - outputs[0, label]

            norm_w = torch.norm(w_k, p=2)

            if norm_w.item() == 0:
                continue

            ratio = torch.abs(f_k) / norm_w

            if ratio < min_ratio:
                min_ratio = ratio
                best_w = w_k.clone()

        if best_w is None:
            break

        norm_w = torch.norm(best_w, p=2)
        
        r_i = (min_ratio + 1e-8) * best_w / norm_w
        
        r_tot += r_i
        
        x_adv = (x + (1 + overshoot) * r_tot).detach()
        
        x_adv = torch.clamp(x_adv, 0, 1)

        itr += 1

    return x_adv.detach()

def weak_attack(x, y, epsilon=0.01, alpha=0.005, steps=20):
    model.eval()
    x_orig = x.clone().detach()
    x_adv  = x_orig.clone()
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        loss = criterion(model(x_adv), y)
        grad = torch.autograd.grad(
            loss,
            x_adv,
            retain_graph=False,
            create_graph=False
        )[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.clamp(x_adv, x_orig - epsilon, x_orig + epsilon).clamp(0, 1)
    return x_adv.detach()


def noise_attack(x, epsilon=0.05, iterations=50):
    model.eval()
    x_orig   = x.clone().detach()
    best_adv = x_orig.clone()
    with torch.no_grad():
        orig_pred = int(torch.argmax(model(x_orig), 1).item())
        best_loss = criterion(model(x_orig), torch.tensor([orig_pred], device=x.device))
        for _ in range(iterations):
            x_cand = torch.clamp(x_orig + torch.randn_like(x_orig) * epsilon, 0, 1)
            loss   = criterion(model(x_cand), torch.tensor([orig_pred], device=x.device))
            if loss > best_loss:
                best_loss = loss
                best_adv  = x_cand
    return best_adv.detach()

# =============================================================================
# 8. ADVERSARIAL TRAINING — SOURCE-INDEPENDENT VALIDATION
# =============================================================================
print(f"\n[Phase 3] Reusing the original {split_unit_name}-level base split ...")

# IMPORTANT:
# Do NOT split X_train again here.
# base_train_ids/base_val_ids were created BEFORE fitting the DNN/RF/SVM,
# and meta_train_ids/meta_val_ids was reserved at the same time. Reusing those indices
# guarantees that no meta source can enter adversarial training.
X_base_train_t = torch.tensor(
    X_base_train, dtype=torch.float32, device=device
)
y_base_train_t = torch.tensor(
    y_base_train, dtype=torch.long, device=device
)
X_clean_val_t = torch.tensor(
    X_base_val, dtype=torch.float32, device=device
)
y_clean_val_t = torch.tensor(
    y_base_val, dtype=torch.long, device=device
)

require_no_overlap(base_train_ids, meta_train_ids, "Base train/meta train overlap")
require_no_overlap(base_train_ids, meta_val_ids, "Base train/meta validation overlap")
require_no_overlap(base_val_ids, meta_train_ids, "Base validation/meta train overlap")
require_no_overlap(base_val_ids, meta_val_ids, "Base validation/meta validation overlap")

print(f"  Base training {split_unit_name}s: {len(X_base_train)}")
print(f"  Clean validation {split_unit_name}s: {len(X_base_val)}")
print(f"  Reserved meta-pool {split_unit_name}s: {len(meta_train_ids) + len(meta_val_ids)}")
print("  Adversarial-training source isolation: PASSED")

require_no_overlap(base_train_ids, meta_train_ids, "Base train/meta train overlap")
require_no_overlap(base_train_ids, meta_val_ids, "Base train/meta validation overlap")
require_no_overlap(base_val_ids, meta_train_ids, "Base validation/meta train overlap")
require_no_overlap(base_val_ids, meta_val_ids, "Base validation/meta validation overlap")
require_no_overlap(train_source_ids, test_source_ids, "Development/final-test overlap")
print("  Final adversarial-training source isolation: PASSED")

print("\n[Phase 3] Generating adversarial examples ...")
model.eval()
n_adv = min(1000, len(X_base_train))
X_adv_list, y_adv_list = [], []
for i in range(n_adv):
    xi, yi = X_base_train_t[i:i+1], y_base_train_t[i:i+1]
    X_adv_list.append(shap_pgd_attack(xi, yi).cpu().numpy())
    X_adv_list.append(deepfool_attack(xi).cpu().numpy())
    X_adv_list.append(weak_attack(xi, yi).cpu().numpy())
    X_adv_list.append(noise_attack(xi).cpu().numpy())
    y_adv_list.extend([yi.item()] * 4)
    if (i + 1) % 50 == 0:
        print(f"  Augmentation {i+1}/{n_adv} done ...")
X_adv = np.vstack(X_adv_list).astype(np.float32)
y_adv = np.array(y_adv_list, dtype=np.int64)
X_train_full_t = torch.tensor(np.vstack([X_base_train, X_adv]), dtype=torch.float32, device=device)
y_train_full_t = torch.tensor(np.concatenate([y_base_train, y_adv]), dtype=torch.long, device=device)
print(f"  Augmented training set: {len(X_train_full_t)} samples")
print(f"  Breakdown: {len(X_base_train)} clean + {len(X_adv)} adversarial (4 attack types × {n_adv})")

print("\n[Phase 4] Adversarial retraining ...")
train_model(model, X_train_full_t, y_train_full_t, X_val_t=X_clean_val_t, y_val_t=y_clean_val_t)
explainer = build_explainer(model, background)
print("  SHAP explainer rebuilt.")

# =============================================================================
# 9. SHAP BASELINE
# FIX: safe_shap ensures consistent vector length == input_dim
# =============================================================================
print("\n[Phase 5] Building SHAP deviation baseline ...")
model.eval()

# IMPORTANT:
# Thresholds/baselines are fitted only from base-training sources.
# Reserved meta-training/meta-validation sources are excluded.
n_baseline = min(1000, len(X_base_train))

baseline_shap = np.stack([
    np.abs(
        safe_shap(
            explainer,
            X_base_train_t[i:i+1],
            input_dim
        )
    )
    for i in range(n_baseline)
])
baseline_mean = baseline_shap.mean(axis=0)   # shape: (input_dim,)
devs     = np.linalg.norm(baseline_shap - baseline_mean, axis=1)
LOW_THR  = float(np.percentile(devs, 90))
HIGH_THR = float(np.percentile(devs, 97))
print(f"  Thresholds  Low (p90): {LOW_THR:.4f}  |  High (p97): {HIGH_THR:.4f}")

# =============================================================================
# 10. GRADIENT SENSITIVITY BASELINE
# FIX: detach().clone() before requires_grad_ avoids in-place leaf error
# =============================================================================
print("\n[Phase 6] Building gradient sensitivity baseline ...")
model.eval()
grad_norms = []
for i in range(min(200, len(X_base_train))):
    xb = (
        X_base_train_t[i:i+1]
        .detach()
        .clone()
        .requires_grad_(True)
    )

    loss = criterion(
        model(xb),
        y_base_train_t[i:i+1]
    )

    grad = torch.autograd.grad(
        loss,
        xb,
        retain_graph=False,
        create_graph=False
    )[0]

    grad_norms.append(
        float(grad.norm().item())
    )

GRAD_MEAN = float(np.mean(grad_norms))
GRAD_STD  = float(np.std(grad_norms))
GRAD_THR  = GRAD_MEAN + 2.5 * GRAD_STD
print(f"  Gradient threshold: {GRAD_THR:.4f}")

# =============================================================================
# 11. FEATURE VARIANCE BASELINE
# =============================================================================
print("\n[Phase 7] Building feature variance baseline ...")
fv = X_base_train[:min(200, len(X_base_train))].var(axis=0)
VAR_THR = float(fv.mean() + 2.5 * fv.std())
print(f"  Feature variance threshold: {VAR_THR:.6f}")

# =============================================================================
# 12. SIGNAL EXTRACTION
# FIX: safe_shap used throughout; aligned lengths for instability computation
# =============================================================================
def extract_signals(x, x_reference=None, n_probes=10):
    """
    Returns a 16-dimensional detector-signal vector.

    x_reference:
        Original clean sample corresponding to x.
        For clean input, x_reference == x.
        For adversarial input, x_reference is the original clean sample.
    """
    model.eval()

    n_feats = x.shape[1]

    # ------------------------------------------------------------
    # Reference sample
    # ------------------------------------------------------------
    if x_reference is None:
        x_reference = x.detach().clone()

    x_reference = x_reference.detach().clone()

    # ------------------------------------------------------------
    # 0. SHAP deviation
    # ------------------------------------------------------------
    sv = safe_shap(explainer, x, n_feats)

    shap_dev = float(
        np.linalg.norm(np.abs(sv) - baseline_mean)
    )

    # ------------------------------------------------------------
    # 1. Gradient norm
    # ------------------------------------------------------------
    xg = x.detach().clone().requires_grad_(True)

    with torch.no_grad():
        lbl = int(torch.argmax(model(x.detach()), 1).item())

    grad_loss = criterion(
        model(xg),
        torch.tensor([lbl], device=x.device)
    )

    grad = torch.autograd.grad(
        grad_loss,
        xg,
        retain_graph=False,
        create_graph=False
    )[0]

    grad_norm = float(grad.norm().item())

    # ------------------------------------------------------------
    # 2 & 3. DNN / Ensemble confidence
    # ------------------------------------------------------------
    with torch.no_grad():
        dnn_probs = torch.softmax(model(x), 1)
        dnn_conf = float(torch.max(dnn_probs).item())

    xnp = x.detach().cpu().numpy()

    rf_conf = float(
        max(rf_model.predict_proba(xnp)[0])
    )

    svm_conf = float(
        max(svm_model.predict_proba(xnp)[0])
    )

    ens_conf = (
        rf_conf + svm_conf + dnn_conf
    ) / 3.0

    # ------------------------------------------------------------
    # 4. SHAP instability
    # ------------------------------------------------------------
    noise = torch.randn_like(x) * 0.001

    sv_noisy = safe_shap(
        explainer,
        (x + noise).clamp(0, 1),
        n_feats
    )

    shap_instab = float(
        np.linalg.norm(sv - sv_noisy)
    )

    # ------------------------------------------------------------
    # 5 & 6. Prediction flip rate + confidence variance
    # ------------------------------------------------------------
    preds_p = []
    confs_p = []

    with torch.no_grad():
        for _ in range(n_probes):
            xn = torch.clamp(
                x + torch.randn_like(x) * 0.01,
                0,
                1
            )

            p = torch.softmax(model(xn), 1)

            preds_p.append(
                int(torch.argmax(p, 1).item())
            )

            confs_p.append(
                float(torch.max(p).item())
            )

    flip_rate = (
        sum(1 for p in preds_p if p != preds_p[0])
        / n_probes
    )

    conf_var = float(np.var(confs_p))

    # ------------------------------------------------------------
    # 7. Logit margin
    # ------------------------------------------------------------
    with torch.no_grad():
        logits_current = model(x)

        sorted_current = torch.sort(
            logits_current,
            descending=True
        ).values

        margin = float(
            (
                sorted_current[0][0]
                - sorted_current[0][1]
            ).item()
        )

    # ============================================================
    # 8-13. PERTURBATION CHARACTERISTICS
    # ============================================================
    delta = (
        x.detach() - x_reference.detach()
    )

    perturbation_l1 = float(
        torch.norm(delta, p=1).item()
    )

    perturbation_l2 = float(
        torch.norm(delta, p=2).item()
    )

    perturbation_linf = float(
        torch.norm(
            delta,
            p=float("inf")
        ).item()
    )

    changed_feature_count = float(
        (
            torch.abs(delta) > 1e-6
        ).sum().item()
    )

    mean_abs_perturbation = float(
        torch.abs(delta).mean().item()
    )

    relative_perturbation = float(
    torch.norm(delta, p=2).item()
    / (torch.norm(x_reference, p=2).item() + 1e-8)
    
    )
    
    # ------------------------------------------------------------
    # 14 & 15. Confidence change + margin change
    # ------------------------------------------------------------
    with torch.no_grad():

        clean_logits = model(x_reference)
        clean_probs = torch.softmax(
            clean_logits, 1
        )

        current_probs = torch.softmax(
            model(x), 1
        )

        clean_conf = float(
            torch.max(clean_probs).item()
        )

        current_conf = float(
            torch.max(current_probs).item()
        )

        confidence_change = (
            current_conf - clean_conf
        )

        clean_sorted = torch.sort(
            clean_logits,
            descending=True
        ).values

        current_sorted = torch.sort(
            model(x),
            descending=True
        ).values

        clean_margin = float(
            (
                clean_sorted[0][0]
                - clean_sorted[0][1]
            ).item()
        )

        current_margin = float(
            (
                current_sorted[0][0]
                - current_sorted[0][1]
            ).item()
        )

        margin_change = (
            current_margin - clean_margin
        )

    # ------------------------------------------------------------
    # RETURN ALL 16 SIGNALS
    # ------------------------------------------------------------
    return np.array([
        shap_dev,
        grad_norm,
        dnn_conf,
        ens_conf,
        shap_instab,
        flip_rate,
        conf_var,
        margin,
        perturbation_l1,
        perturbation_l2,
        perturbation_linf,
        changed_feature_count,
        mean_abs_perturbation,
        relative_perturbation,
        confidence_change,
        margin_change
    ], dtype=np.float32)

SIGNAL_NAMES = [
    "shap_dev",
    "grad_norm",
    "dnn_confidence",
    "ensemble_conf",
    "shap_instability",
    "pred_flip_rate",
    "conf_variance",
    "logit_margin",
    "perturbation_l1",
    "perturbation_l2",
    "perturbation_linf",
    "changed_feature_count",
    "mean_abs_perturbation",
    "relative_perturbation",
    "confidence_change",
    "margin_change"
]

# =============================================================================
# 13. META-DATASET — STRICT SPLIT ISOLATION
# =============================================================================
META_LABEL_NAMES = {
    0: "Clean",
    1: "SHAP-PGD Attack",
    2: "DeepFool Attack",
    3: "Weak Attack",
    4: "Noise Attack"
}

print(f"\n[Phase 8] Building strictly {split_unit_name}-independent meta-dataset ...")

# Meta-training and meta-validation were separated BEFORE preprocessing fit.
# Both are transformed only with preprocessing fitted on base training.
print(f"  Meta-training {split_unit_name}s:   {len(X_meta_train)}")
print(f"  Meta-validation {split_unit_name}s: {len(X_meta_val)}")

require_no_overlap(meta_train_ids, meta_val_ids, "Meta train/meta validation overlap")
require_no_overlap(meta_train_ids, base_train_ids, "Meta train/base train overlap")
require_no_overlap(meta_val_ids, base_train_ids, "Meta validation/base train overlap")
require_no_overlap(meta_train_ids, base_val_ids, "Meta train/base validation overlap")
require_no_overlap(meta_val_ids, base_val_ids, "Meta validation/base validation overlap")

print("  Meta source isolation: PASSED")


def build_meta_group(X_sources_t, y_sources_t):
    """
    Build five detector-signal examples for each source.

    The original clean sample is used as x_reference
    for all adversarial variants.
    """

    X_list = []
    y_list = []

    for count in range(len(X_sources_t)):

        xi = X_sources_t[count:count+1]
        yi = y_sources_t[count:count+1]

        # --------------------------------------------------------
        # Clean
        # --------------------------------------------------------
        X_list.append(
            extract_signals(
                xi,
                xi
            )
        )
        y_list.append(0)

        # --------------------------------------------------------
        # SHAP-PGD
        # --------------------------------------------------------
        x_shap = shap_pgd_attack(
            xi,
            yi
        )

        X_list.append(
            extract_signals(
                x_shap,
                xi
            )
        )
        y_list.append(1)

        # --------------------------------------------------------
        # DeepFool
        # --------------------------------------------------------
        x_df = deepfool_attack(
            xi
        )

        X_list.append(
            extract_signals(
                x_df,
                xi
            )
        )
        y_list.append(2)

        # --------------------------------------------------------
        # Weak PGD
        # --------------------------------------------------------
        x_weak = weak_attack(
            xi,
            yi
        )

        X_list.append(
            extract_signals(
                x_weak,
                xi
            )
        )
        y_list.append(3)

        # --------------------------------------------------------
        # Noise
        # --------------------------------------------------------
        x_noise = noise_attack(
            xi
        )

        X_list.append(
            extract_signals(
                x_noise,
                xi
            )
        )
        y_list.append(4)

        if (count + 1) % 50 == 0:
            print(
                f"  Processed {count+1}/{len(X_sources_t)}"
            )

    return (
        np.vstack(X_list),
        np.array(y_list, dtype=np.int64)
    )


print("\n  Building meta-training group ...")

meta_X_tr, meta_y_tr = build_meta_group(
    X_meta_train_t,
    y_meta_train_t
)

print("\n  Building meta-validation group ...")

meta_X_val, meta_y_val = build_meta_group(
    X_meta_val_t,
    y_meta_val_t
)

print(
    f"\n  Meta-training set:   "
    f"{meta_X_tr.shape[0]} samples | "
    f"{meta_X_tr.shape[1]} signals"
)

print(
    f"  Meta-validation set: "
    f"{meta_X_val.shape[0]} samples | "
    f"{meta_X_val.shape[1]} signals"
)

# =============================================================================
# 14. META-CLASSIFIER
# =============================================================================
print(f"\n[Phase 9] Training {split_unit_name}-independent meta-classifier ...")

meta_model = GradientBoostingClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=3,
    min_samples_leaf=5,
    subsample=0.9,
    random_state=42
)

meta_model.fit(
    meta_X_tr,
    meta_y_tr
)

mp = meta_model.predict(
    meta_X_val
)

print(
    f"  Source-independent validation accuracy: "
    f"{accuracy_score(meta_y_val, mp):.4f}"
)

print(
    classification_report(
        meta_y_val,
        mp,
        target_names=[
            META_LABEL_NAMES[i]
            for i in range(5)
        ]
    )
)

print("  Signal importances:")

for name, imp in sorted(
    zip(
        SIGNAL_NAMES,
        meta_model.feature_importances_
    ),
    key=lambda t: -t[1]
):
    print(
        f"    {name:20s} "
        f"{imp:.4f} "
        f"{'#' * int(imp * 60)}"
    )

# =============================================================================

# =============================================================================
# CHECKPOINT SAVE — RUN 1 COMPLETE
# =============================================================================
import os
import joblib

ARTIFACT_DIR = "/kaggle/working/hybrid_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save the final DNN weights/configuration.
torch.save(
    {
        "state_dict": model.state_dict(),
        "input_dim": int(input_dim),
        "num_classes": int(num_classes),
    },
    os.path.join(ARTIFACT_DIR, "dnn_model.pt")
)

# Save sklearn models.
joblib.dump(rf_model, os.path.join(ARTIFACT_DIR, "rf_model.joblib"))
joblib.dump(svm_model, os.path.join(ARTIFACT_DIR, "svm_model.joblib"))
joblib.dump(meta_model, os.path.join(ARTIFACT_DIR, "meta_model.joblib"))
joblib.dump(scaler, os.path.join(ARTIFACT_DIR, "scaler.joblib"))
if onehot_encoder is not None:
    joblib.dump(onehot_encoder, os.path.join(ARTIFACT_DIR, "onehot_encoder.joblib"))

# Save all data needed for chunked final evaluation.
np.savez_compressed(
    os.path.join(ARTIFACT_DIR, "evaluation_data.npz"),
    X_test=X_test,
    y_test=y_test,
    background=background.detach().cpu().numpy(),
    baseline_mean=baseline_mean.astype(np.float32),
    X_base_train_shape=np.array(X_base_train.shape, dtype=np.int64),
    input_dim=np.array([input_dim], dtype=np.int64),
    num_classes=np.array([num_classes], dtype=np.int64),
    GRAD_THR=np.array([GRAD_THR], dtype=np.float32),
    VAR_THR=np.array([VAR_THR], dtype=np.float32),
    LOW_THR=np.array([LOW_THR], dtype=np.float32),
    HIGH_THR=np.array([HIGH_THR], dtype=np.float32),
)

metadata = {
    "dataset": "X-IIoTID",
    "label_col": label_col,
    "binary_label_mapping": BINARY_LABEL_MAPPING,
    "source_col": source_col,
    "split_unit_name": split_unit_name,
    "n_meta": int(n_meta),
    "n_test": int(len(X_test)),
    "n_adv": int(n_adv),
    "attack_parameters": {
        "shap_pgd": {"epsilon": 0.1, "alpha": 0.02, "steps": 3, "top_k": 5},
        "deepfool": {"overshoot": 0.02, "max_iter": 20},
        "weak": {"epsilon": 0.01, "alpha": 0.005, "steps": 20},
        "noise": {"epsilon": 0.05, "iterations": 50},
    },
    "signal_parameters": {"n_probes": 10, "signal_count": 16},
    "meta_output": {
        0: None,
        1: "SHAP-PGD Attack",
        2: "DeepFool Attack",
        3: "Weak Adversarial Attack",
        4: "Noise-Based Adversarial Attack",
    },
}
joblib.dump(metadata, os.path.join(ARTIFACT_DIR, "metadata.joblib"))

print("\n" + "=" * 80)
print("RUN 1 COMPLETE — TRAINING ARTIFACTS SAVED")
print("=" * 80)
print(f"Saved to: {ARTIFACT_DIR}")
print("Files:")
for name in sorted(os.listdir(ARTIFACT_DIR)):
    print(f"  - {name}")
print("\nSTOP HERE. Do NOT run final evaluation in Run 1.")


import os
import shutil

artifact_dir = "/kaggle/working/hybrid_artifacts"

print("Exists:", os.path.isdir(artifact_dir))

if os.path.isdir(artifact_dir):
    print("\nFiles:")
    for f in sorted(os.listdir(artifact_dir)):
        print(" -", f)

    zip_path = shutil.make_archive(
        "/kaggle/working/hybrid_artifacts_backup",
        "zip",
        root_dir="/kaggle/working",
        base_dir="hybrid_artifacts"
    )

    print("\nBACKUP CREATED:")
    print(zip_path)
else:
    print("ERROR: artifacts are not present in this session.")


import os
import time
import json
import joblib
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn

# =============================================================================
# RUN 2 — CHUNKED FINAL EVALUATION
# Edit CHUNK_ID = 0 to run chunks 0..9.
# Each chunk processes up to 1,000 FINAL TEST samples.
# =============================================================================
CHUNK_ID = 0
CHUNK_SIZE = 1000
ARTIFACT_DIR = ""
RESULT_DIR = "/kaggle/working/hybrid_eval_chunks"
os.makedirs(RESULT_DIR, exist_ok=True)

if not (0 <= CHUNK_ID <= 9):
    raise ValueError("CHUNK_ID must be an integer from 0 to 9.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------------------------------------------------------
# LOAD CHECKPOINTS FROM RUN 1
# -----------------------------------------------------------------------------
required = [
    "dnn_model.pt",
    "rf_model.joblib",
    "svm_model.joblib",
    "meta_model.joblib",
    "evaluation_data.npz",
    "metadata.joblib",
]
missing = [f for f in required if not os.path.exists(os.path.join(ARTIFACT_DIR, f))]
if missing:
    raise FileNotFoundError(
        "Run 1 artifacts are missing: " + ", ".join(missing)
    )

meta = joblib.load(os.path.join(ARTIFACT_DIR, "metadata.joblib"))
rf_model = joblib.load(os.path.join(ARTIFACT_DIR, "rf_model.joblib"))
svm_model = joblib.load(os.path.join(ARTIFACT_DIR, "svm_model.joblib"))
meta_model = joblib.load(os.path.join(ARTIFACT_DIR, "meta_model.joblib"))
arr = np.load(os.path.join(ARTIFACT_DIR, "evaluation_data.npz"))

X_test = arr["X_test"].astype(np.float32)
y_test = arr["y_test"].astype(np.int64)
background_np = arr["background"].astype(np.float32)
baseline_mean = arr["baseline_mean"].astype(np.float32)
input_dim = int(arr["input_dim"][0])
num_classes = int(arr["num_classes"][0])
GRAD_THR = float(arr["GRAD_THR"][0])
VAR_THR = float(arr["VAR_THR"][0])
LOW_THR = float(arr["LOW_THR"][0])
HIGH_THR = float(arr["HIGH_THR"][0])

X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

# -----------------------------------------------------------------------------
# MODEL
# -----------------------------------------------------------------------------
class NIDSModel(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

checkpoint = torch.load(
    os.path.join(ARTIFACT_DIR, "dnn_model.pt"),
    map_location=device,
)
model = NIDSModel(input_dim, num_classes).to(device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
criterion = nn.CrossEntropyLoss()

background = torch.tensor(background_np, dtype=torch.float32, device=device)
explainer = shap.DeepExplainer(model, background.clone().detach())

# -----------------------------------------------------------------------------
# SHAP UTILITIES — same logic as the original script
# -----------------------------------------------------------------------------
def get_shap_vector(explainer, x, n_feats, label_idx=1):
    shap_vals = explainer.shap_values(x.detach(), check_additivity=False)
    sample_count = x.shape[0]

    if isinstance(shap_vals, list):
        idx = min(label_idx, len(shap_vals) - 1)
        arr = np.array(shap_vals[idx])
        if arr.ndim >= 2 and arr.shape[0] == sample_count:
            arr = arr[0]
        return arr.squeeze()

    arr = np.array(shap_vals)
    if arr.ndim == 3:
        if arr.shape[0] == sample_count and arr.shape[1] == n_feats:
            class_idx = min(label_idx, arr.shape[2] - 1)
            arr = arr[0, :, class_idx]
        elif arr.shape[1] == sample_count and arr.shape[2] == n_feats:
            class_idx = min(label_idx, arr.shape[0] - 1)
            arr = arr[class_idx, 0, :]
        elif arr.shape[0] == sample_count and arr.shape[2] == n_feats:
            class_idx = min(label_idx, arr.shape[1] - 1)
            arr = arr[0, class_idx, :]
    elif arr.ndim == 2 and arr.shape[0] == sample_count:
        arr = arr[0]
    return arr.squeeze()


def safe_shap(explainer, x, n_feats):
    vec = np.atleast_1d(
        get_shap_vector(explainer, x, n_feats, label_idx=1)
    ).flatten()
    if len(vec) > n_feats:
        vec = vec[:n_feats]
    elif len(vec) < n_feats:
        vec = np.pad(vec, (0, n_feats - len(vec)))
    return vec


def safe_shap_for_label(explainer, x, n_feats, y):
    label_idx = int(y.item()) if torch.is_tensor(y) else int(y)
    vec = np.atleast_1d(
        get_shap_vector(explainer, x, n_feats, label_idx=label_idx)
    ).flatten()
    if len(vec) > n_feats:
        vec = vec[:n_feats]
    elif len(vec) < n_feats:
        vec = np.pad(vec, (0, n_feats - len(vec)))
    return vec

# -----------------------------------------------------------------------------
# ATTACKS — same settings as original final experiment
# -----------------------------------------------------------------------------
def shap_pgd_attack(x, y, epsilon=0.1, alpha=0.02, steps=3, top_k=5):
    model.eval()
    x_orig = x.clone().detach()
    x_adv = x_orig.clone()
    n_feats = x_orig.shape[1]
    shap_vec = safe_shap_for_label(explainer, x_orig, n_feats, y)
    important = np.argsort(-np.abs(shap_vec))[:top_k]
    mask = torch.zeros_like(x_orig)
    mask[0, important] = 1.0
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        loss = criterion(model(x_adv), y)
        grad = torch.autograd.grad(
            loss, x_adv, retain_graph=False, create_graph=False
        )[0]
        with torch.no_grad():
            x_adv = x_adv.detach() + alpha * grad.sign() * mask
            x_adv = torch.clamp(x_adv, x_orig - epsilon, x_orig + epsilon)
            x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()


def deepfool_attack(x, overshoot=0.02, max_iter=20):
    model.eval()
    x_adv = x.clone().detach().requires_grad_(True)
    with torch.no_grad():
        logits = model(x_adv)
        label = logits.argmax(1).item()
    num_classes = logits.shape[1]
    r_tot = torch.zeros_like(x_adv)
    itr = 0
    while itr < max_iter:
        x_adv.requires_grad_(True)
        outputs = model(x_adv)
        current = outputs.argmax(1).item()
        if current != label:
            break
        min_ratio = float("inf")
        best_w = None
        for k in range(num_classes):
            if k == label:
                continue
            model.zero_grad()
            grad_k = torch.autograd.grad(
                outputs[0, k], x_adv, retain_graph=True
            )[0]
            grad_label = torch.autograd.grad(
                outputs[0, label], x_adv, retain_graph=True
            )[0]
            w_k = grad_k - grad_label
            f_k = outputs[0, k] - outputs[0, label]
            norm_w = torch.norm(w_k, p=2)
            if norm_w.item() == 0:
                continue
            ratio = torch.abs(f_k) / norm_w
            if ratio < min_ratio:
                min_ratio = ratio
                best_w = w_k.clone()
        if best_w is None:
            break
        norm_w = torch.norm(best_w, p=2)
        r_i = (min_ratio + 1e-8) * best_w / norm_w
        r_tot += r_i
        x_adv = (x + (1 + overshoot) * r_tot).detach()
        x_adv = torch.clamp(x_adv, 0, 1)
        itr += 1
    return x_adv.detach()


def weak_attack(x, y, epsilon=0.01, alpha=0.005, steps=20):
    model.eval()
    x_orig = x.clone().detach()
    x_adv = x_orig.clone()
    for _ in range(steps):
        x_adv = x_adv.detach().requires_grad_(True)
        loss = criterion(model(x_adv), y)
        grad = torch.autograd.grad(
            loss, x_adv, retain_graph=False, create_graph=False
        )[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.clamp(
                x_adv, x_orig - epsilon, x_orig + epsilon
            ).clamp(0, 1)
    return x_adv.detach()


def noise_attack(x, epsilon=0.05, iterations=50):
    model.eval()
    x_orig = x.clone().detach()
    best_adv = x_orig.clone()
    with torch.no_grad():
        orig_pred = int(torch.argmax(model(x_orig), 1).item())
        target = torch.tensor([orig_pred], device=x.device)
        best_loss = criterion(model(x_orig), target)
        for _ in range(iterations):
            x_cand = torch.clamp(
                x_orig + torch.randn_like(x_orig) * epsilon, 0, 1
            )
            loss = criterion(model(x_cand), target)
            if loss > best_loss:
                best_loss = loss
                best_adv = x_cand
    return best_adv.detach()

# -----------------------------------------------------------------------------
# 16-SIGNAL EXTRACTION — same logic as original script
# -----------------------------------------------------------------------------
def extract_signals(x, x_reference=None, n_probes=10):
    model.eval()
    n_feats = x.shape[1]
    if x_reference is None:
        x_reference = x.detach().clone()
    x_reference = x_reference.detach().clone()

    sv = safe_shap(explainer, x, n_feats)
    shap_dev = float(np.linalg.norm(np.abs(sv) - baseline_mean))

    xg = x.detach().clone().requires_grad_(True)
    with torch.no_grad():
        lbl = int(torch.argmax(model(x.detach()), 1).item())
    grad_loss = criterion(
        model(xg), torch.tensor([lbl], device=x.device)
    )
    grad = torch.autograd.grad(
        grad_loss, xg, retain_graph=False, create_graph=False
    )[0]
    grad_norm = float(grad.norm().item())

    with torch.no_grad():
        dnn_probs = torch.softmax(model(x), 1)
        dnn_conf = float(torch.max(dnn_probs).item())
    xnp = x.detach().cpu().numpy()
    rf_conf = float(max(rf_model.predict_proba(xnp)[0]))
    svm_conf = float(max(svm_model.predict_proba(xnp)[0]))
    ens_conf = (rf_conf + svm_conf + dnn_conf) / 3.0

    noise = torch.randn_like(x) * 0.001
    sv_noisy = safe_shap(explainer, (x + noise).clamp(0, 1), n_feats)
    shap_instab = float(np.linalg.norm(sv - sv_noisy))

    preds_p, confs_p = [], []
    with torch.no_grad():
        for _ in range(n_probes):
            xn = torch.clamp(
                x + torch.randn_like(x) * 0.01, 0, 1
            )
            p = torch.softmax(model(xn), 1)
            preds_p.append(int(torch.argmax(p, 1).item()))
            confs_p.append(float(torch.max(p).item()))
    flip_rate = sum(1 for p in preds_p if p != preds_p[0]) / n_probes
    conf_var = float(np.var(confs_p))

    with torch.no_grad():
        logits_current = model(x)
        sorted_current = torch.sort(logits_current, descending=True).values
        margin = float(
            (sorted_current[0][0] - sorted_current[0][1]).item()
        )

    delta = x.detach() - x_reference.detach()
    perturbation_l1 = float(torch.norm(delta, p=1).item())
    perturbation_l2 = float(torch.norm(delta, p=2).item())
    perturbation_linf = float(torch.norm(delta, p=float("inf")).item())
    changed_feature_count = float((torch.abs(delta) > 1e-6).sum().item())
    mean_abs_perturbation = float(torch.abs(delta).mean().item())
    max_abs_perturbation = float(torch.abs(delta).max().item())

    with torch.no_grad():
        clean_logits = model(x_reference)
        clean_probs = torch.softmax(clean_logits, 1)
        current_probs = torch.softmax(model(x), 1)
        clean_conf = float(torch.max(clean_probs).item())
        current_conf = float(torch.max(current_probs).item())
        confidence_change = current_conf - clean_conf
        clean_sorted = torch.sort(clean_logits, descending=True).values
        current_sorted = torch.sort(model(x), descending=True).values
        clean_margin = float(
            (clean_sorted[0][0] - clean_sorted[0][1]).item()
        )
        current_margin = float(
            (current_sorted[0][0] - current_sorted[0][1]).item()
        )
        margin_change = current_margin - clean_margin

    return np.array([
        shap_dev, grad_norm, dnn_conf, ens_conf,
        shap_instab, flip_rate, conf_var, margin,
        perturbation_l1, perturbation_l2, perturbation_linf,
        changed_feature_count, mean_abs_perturbation,
        max_abs_perturbation, confidence_change, margin_change
    ], dtype=np.float32)

# -----------------------------------------------------------------------------
# DETECTOR
# -----------------------------------------------------------------------------
META_OUTPUT = {
    0: None,
    1: "SHAP-PGD Attack",
    2: "DeepFool Attack",
    3: "Weak Adversarial Attack",
    4: "Noise-Based Adversarial Attack",
}

ATTACK_LABELS = {
    "SHAP-PGD Attack",
    "DeepFool Attack",
    "Weak Adversarial Attack",
    "Noise-Based Adversarial Attack",
}


def detect_all_hybrid(x, x_reference=None):
    signals = extract_signals(x, x_reference).reshape(1, -1)

    # Predicted class
    meta_pred = int(meta_model.predict(signals)[0])
    predicted_label = META_OUTPUT[meta_pred]

    # Probability for every meta-classifier class
    meta_probs = meta_model.predict_proba(signals)[0]

    # Make sure probabilities are aligned with classes 0,1,2,3,4
    prob_by_class = {
        int(cls): float(prob)
        for cls, prob in zip(meta_model.classes_, meta_probs)
    }

    return predicted_label, prob_by_class


def shap_rule(x):
    sv = safe_shap(explainer, x, input_dim)
    dev = float(np.linalg.norm(np.abs(sv) - baseline_mean))
    return dev > HIGH_THR, dev > LOW_THR


def confidence_rule(x):
    model.eval()
    with torch.no_grad():
        return float(torch.max(torch.softmax(model(x), 1)).item()) < 0.6


def ensemble_rule(x):
    xnp = x.detach().cpu().numpy()
    rf = float(max(rf_model.predict_proba(xnp)[0]))
    svm = float(max(svm_model.predict_proba(xnp)[0]))
    with torch.no_grad():
        dnn = float(torch.max(torch.softmax(model(x), 1)).item())
    return (rf + svm + dnn) / 3.0 < 0.6


def detect_all_rules(x):
    model.eval()
    xg = x.detach().clone().requires_grad_(True)
    with torch.no_grad():
        lbl = int(torch.argmax(model(x.detach()), 1).item())
    grad_loss = criterion(
        model(xg), torch.tensor([lbl], device=x.device)
    )
    grad = torch.autograd.grad(
        grad_loss, xg, retain_graph=False, create_graph=False
    )[0]
    if float(grad.norm().item()) > GRAD_THR:
        return "Weak Adversarial Attack"

    preds_p, confs_p = [], []
    with torch.no_grad():
        for _ in range(10):
            xn = torch.clamp(x + torch.randn_like(x) * 0.01, 0, 1)
            p = torch.softmax(model(xn), 1)
            preds_p.append(int(torch.argmax(p, 1).item()))
            confs_p.append(float(torch.max(p).item()))

    if (
        sum(1 for p in preds_p if p != preds_p[0]) / 10 > 0.3
        or np.var(confs_p) > 0.02
        or x.detach().cpu().numpy().var() > VAR_THR
    ):
        return "Noise-Based Adversarial Attack"

    is_high, is_low = shap_rule(x)
    if is_high:
        return "SHAP-PGD Attack"
    if is_low and (confidence_rule(x) or ensemble_rule(x)):
        return "SHAP-PGD Attack"
    if is_low:
        return "Suspicious"

    with torch.no_grad():
        pred = int(torch.argmax(model(x), 1).item())
    return "Benign" if pred == 1 else "Unknown Attack"

# -----------------------------------------------------------------------------
# RESUMABLE CHUNK EXECUTION
# -----------------------------------------------------------------------------
start = CHUNK_ID * CHUNK_SIZE
end = min(start + CHUNK_SIZE, len(X_test))
if start >= len(X_test):
    raise ValueError(
        f"CHUNK_ID={CHUNK_ID} starts at {start}, but final test has only {len(X_test)} samples."
    )

result_path = os.path.join(RESULT_DIR, f"chunk_{CHUNK_ID:02d}.joblib")
if os.path.exists(result_path):
    print(f"Chunk already exists: {result_path}")
    print("Delete it only if you intentionally want to recompute this chunk.")
    raise SystemExit(0)

print("=" * 80)
print(f"RUN 2 — EVALUATING CHUNK {CHUNK_ID}/9")
print(f"Test rows: {start} to {end-1} ({end-start} samples)")
print(f"Output: {result_path}")
print("=" * 80)

results_hybrid = []
results_rules = []
t0 = time.time()

for local_i, i in enumerate(range(start, end), start=1):
    xi = X_test_t[i:i+1]
    yi = torch.tensor([y_test[i]], dtype=torch.long, device=device)

    row_h = {"Test Index": i, "True Label": int(y_test[i])}
    row_r = {"Test Index": i, "True Label": int(y_test[i])}

    attacks = [
        ("Clean", xi),
        ("SHAP-PGD", shap_pgd_attack(xi, yi)),
        ("DeepFool", deepfool_attack(xi)),
        ("Weak", weak_attack(xi, yi)),
        ("Noise", noise_attack(xi)),
    ]

    for col, xadv in attacks:

        # Hybrid detector prediction + probabilities
        hybrid_pred, hybrid_probs = detect_all_hybrid(xadv, xi)

        row_h[col] = hybrid_pred

        # Save meta-classifier probabilities
        row_h[f"{col}_Prob_None"] = hybrid_probs.get(0, 0.0)
        row_h[f"{col}_Prob_SHAP-PGD"] = hybrid_probs.get(1, 0.0)
        row_h[f"{col}_Prob_DeepFool"] = hybrid_probs.get(2, 0.0)
        row_h[f"{col}_Prob_Weak"] = hybrid_probs.get(3, 0.0)
        row_h[f"{col}_Prob_Noise"] = hybrid_probs.get(4, 0.0)

        # Rule-based detector
        row_r[col] = detect_all_rules(xadv)

    results_hybrid.append(row_h)
    results_rules.append(row_r)

    if local_i == 1 or local_i % 25 == 0 or local_i == (end-start):
        elapsed = time.time() - t0
        rate = local_i / max(elapsed, 1e-9)
        remaining = (end-start-local_i) / max(rate, 1e-9)
        print(
            f"  {local_i:4d}/{end-start} | "
            f"elapsed={elapsed/60:.1f} min | "
            f"ETA={remaining/60:.1f} min"
        )

df_hybrid = pd.DataFrame(results_hybrid)
df_rules = pd.DataFrame(results_rules)

payload = {
    "chunk_id": CHUNK_ID,
    "start": start,
    "end": end,
    "n_samples": end - start,
    "df_hybrid": df_hybrid,
    "df_rules": df_rules,
    "metadata": meta,
    "elapsed_seconds": time.time() - t0,
}
joblib.dump(payload, result_path, compress=3)

print("\n" + "=" * 80)
print(f"CHUNK {CHUNK_ID} SAVED")
print(f"Rows: {end-start}")
print(f"Time: {payload['elapsed_seconds']/3600:.2f} hours")
print(f"File: {result_path}")
print("=" * 80)


import os
import joblib
import pandas as pd

RESULT_DIR = "/kaggle/working/hybrid_eval_chunks"

hybrid_parts = []
rules_parts = []

for i in range(10):
    path = os.path.join(RESULT_DIR, f"chunk_{i:02d}.joblib")
    
    data = joblib.load(path)
    
    hybrid_parts.append(data["df_hybrid"])
    rules_parts.append(data["df_rules"])

df_hybrid_all = pd.concat(hybrid_parts, ignore_index=True)
df_rules_all = pd.concat(rules_parts, ignore_index=True)

print("Hybrid rows:", len(df_hybrid_all))
print("Rule rows:", len(df_rules_all))

display(df_hybrid_all.head())


combined_path = "/kaggle/working/run2_combined_results.joblib"

joblib.dump(
    {
        "df_hybrid": df_hybrid_all,
        "df_rules": df_rules_all
    },
    combined_path,
    compress=3
)

print("Saved:", combined_path)

import os
import glob
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

ARTIFACT_DIR = "/kaggle/input/datasets/venusaireddyalla/artifacts/hybrid_artifacts"
RESULT_DIR = "/kaggle/working/hybrid_eval_chunks"

required = [
    "evaluation_data.npz",
    "metadata.joblib",
]
missing = [f for f in required if not os.path.exists(os.path.join(ARTIFACT_DIR, f))]
if missing:
    raise FileNotFoundError("Missing Run 1 artifacts: " + ", ".join(missing))

arr = np.load(os.path.join(ARTIFACT_DIR, "evaluation_data.npz"))
X_test = arr["X_test"].astype(np.float32)
y_test = arr["y_test"].astype(np.int64)

files = sorted(glob.glob(os.path.join(RESULT_DIR, "chunk_*.joblib")))
if len(files) != 10:
    raise RuntimeError(
        f"Expected 10 chunk files (chunk_00..chunk_09), found {len(files)}. "
        "Run all chunks before combining."
    )

payloads = [joblib.load(f) for f in files]
payloads.sort(key=lambda p: p["chunk_id"])

expected_start = 0
combined_h = []
combined_r = []

for p in payloads:
    if p["start"] != expected_start:
        raise RuntimeError(
            f"Chunk ordering/gap detected: expected start {expected_start}, "
            f"got {p['start']} for chunk {p['chunk_id']}."
        )
    combined_h.append(p["df_hybrid"])
    combined_r.append(p["df_rules"])
    expected_start = p["end"]

if expected_start != min(10000, len(X_test)):
    raise RuntimeError(
        f"Chunks cover {expected_start} samples, but expected "
        f"{min(10000, len(X_test))}."
    )

df_hybrid = pd.concat(combined_h, ignore_index=True)
df_rules = pd.concat(combined_r, ignore_index=True)

n_eval = len(df_hybrid)

# -----------------------------------------------------------------------------
# EXACT FOUR-ATTACK-TYPE ROC-AUC
# -----------------------------------------------------------------------------

n_eval = len(df_hybrid)

attack_names = ["SHAP-PGD", "DeepFool", "Weak", "Noise"]

# -------------------------------------------------------------------------
# Build probability matrix
#
# Each block contains n_eval samples generated by one attack.
# For each sample, we retain the probability assigned by the hybrid
# meta-classifier to each of the four attack classes.
# -------------------------------------------------------------------------

prob_matrix = np.vstack([

    # SHAP-PGD attack samples
    df_hybrid[
        [
            "SHAP-PGD_Prob_SHAP-PGD",
            "SHAP-PGD_Prob_DeepFool",
            "SHAP-PGD_Prob_Weak",
            "SHAP-PGD_Prob_Noise",
        ]
    ].values,

    # DeepFool attack samples
    df_hybrid[
        [
            "DeepFool_Prob_SHAP-PGD",
            "DeepFool_Prob_DeepFool",
            "DeepFool_Prob_Weak",
            "DeepFool_Prob_Noise",
        ]
    ].values,

    # Weak attack samples
    df_hybrid[
        [
            "Weak_Prob_SHAP-PGD",
            "Weak_Prob_DeepFool",
            "Weak_Prob_Weak",
            "Weak_Prob_Noise",
        ]
    ].values,

    # Noise attack samples
    df_hybrid[
        [
            "Noise_Prob_SHAP-PGD",
            "Noise_Prob_DeepFool",
            "Noise_Prob_Weak",
            "Noise_Prob_Noise",
        ]
    ].values,
])

# -------------------------------------------------------------------------
# True attack-type labels
# 0 = SHAP-PGD
# 1 = DeepFool
# 2 = Weak
# 3 = Noise
# -------------------------------------------------------------------------

y_true_attack = np.concatenate([
    np.full(n_eval, 0, dtype=int),
    np.full(n_eval, 1, dtype=int),
    np.full(n_eval, 2, dtype=int),
    np.full(n_eval, 3, dtype=int),
])

# -------------------------------------------------------------------------
# Normalize the four attack probabilities.
#
# The original meta-classifier has five classes:
# 0 = None
# 1 = SHAP-PGD
# 2 = DeepFool
# 3 = Weak
# 4 = Noise
#
# For exact attack-type identification, this experiment conditions on
# an input already being one of the four adversarial attack types.
# Therefore, the four attack probabilities are renormalized.
# -------------------------------------------------------------------------

row_sums = prob_matrix.sum(axis=1, keepdims=True)

prob_matrix = (
    prob_matrix /
    np.maximum(row_sums, 1e-12)
)

# -------------------------------------------------------------------------
# Overall multiclass ROC-AUC — One-vs-Rest
# -------------------------------------------------------------------------

exact_attack_auc_macro = roc_auc_score(
    y_true_attack,
    prob_matrix,
    multi_class="ovr",
    average="macro"
)

exact_attack_auc_weighted = roc_auc_score(
    y_true_attack,
    prob_matrix,
    multi_class="ovr",
    average="weighted"
)

# -------------------------------------------------------------------------
# Per-attack ROC-AUC
# -------------------------------------------------------------------------

from sklearn.preprocessing import label_binarize

y_true_onehot = label_binarize(
    y_true_attack,
    classes=[0, 1, 2, 3]
)

attack_auc_rows = []

for i, attack in enumerate(attack_names):

    auc_i = roc_auc_score(
        y_true_onehot[:, i],
        prob_matrix[:, i]
    )

    attack_auc_rows.append({
        "Attack Type": attack,
        "ROC-AUC": auc_i
    })

print("\n" + "=" * 90)
print("EXACT FOUR-ATTACK-TYPE ROC-AUC")
print("=" * 90)

for row in attack_auc_rows:
    print(
        f"{row['Attack Type']:12s}: "
        f"{row['ROC-AUC']:.4f}"
    )

print("-" * 90)
print(
    f"Macro ROC-AUC (OvR)    : "
    f"{exact_attack_auc_macro:.4f}"
)

print(
    f"Weighted ROC-AUC (OvR) : "
    f"{exact_attack_auc_weighted:.4f}"
)

# -----------------------------------------------------------------------------
# DNN BINARY METRICS ON CLEAN FINAL TEST SET
# -----------------------------------------------------------------------------
# Reconstruct DNN probabilities from saved model.
import torch
import torch.nn as nn

class NIDSModel(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(
    os.path.join(ARTIFACT_DIR, "dnn_model.pt"),
    map_location=device,
)
model = NIDSModel(int(arr["input_dim"][0]), int(arr["num_classes"][0])).to(device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1).cpu().numpy()

true_labels = y_test
cm = confusion_matrix(true_labels, preds, labels=[0, 1])
TN = cm[1, 1]
FP = cm[1, 0]
FN = cm[0, 1]
TP = cm[0, 0]

accuracy = accuracy_score(true_labels, preds)
precision = precision_score(true_labels, preds, pos_label=0, zero_division=0)
recall = recall_score(true_labels, preds, pos_label=0, zero_division=0)
f1 = f1_score(true_labels, preds, pos_label=0, zero_division=0)
fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
attack_detection_rate = TP / (TP + FN) if (TP + FN) > 0 else 0.0
attack_true = (true_labels == 0).astype(int)
attack_probs = probs[:, 0].cpu().numpy()
auc = roc_auc_score(attack_true, attack_probs) if len(np.unique(attack_true)) == 2 else float("nan")

# -----------------------------------------------------------------------------
# EXACT FOUR-ATTACK TYPE METRICS — HYBRID
# -----------------------------------------------------------------------------
attack_columns = {
    "SHAP-PGD": "SHAP-PGD Attack",
    "DeepFool": "DeepFool Attack",
    "Weak": "Weak Adversarial Attack",
    "Noise": "Noise-Based Adversarial Attack",
}
attack_names = list(attack_columns.keys())

actual_attack_types = []
predicted_attack_types = []

for attack_name in attack_names:
    actual_attack_types.extend([attack_name] * n_eval)
    predicted_attack_types.extend(df_hybrid[attack_name].values)

actual_attack_types = np.array(actual_attack_types)
predicted_attack_types = np.array(predicted_attack_types)

label_to_class = {
    "SHAP-PGD Attack": "SHAP-PGD",
    "DeepFool Attack": "DeepFool",
    "Weak Adversarial Attack": "Weak",
    "Noise-Based Adversarial Attack": "Noise",
}

predicted_attack_classes = np.array([
    label_to_class.get(x, "Other")
    for x in predicted_attack_types
])

exact_accuracy = accuracy_score(
    actual_attack_types,
    predicted_attack_classes,
)

print("\n" + "=" * 90)
print("FINAL COMBINED RESULTS — HYBRID DETECTOR")
print("=" * 90)
print(f"Number of combined test samples per attack: {n_eval}")
print(f"Total adversarial attack instances evaluated: {4*n_eval}")

print("\nDNN BINARY CLASSIFICATION — CLEAN FINAL TEST")
print(f"Accuracy                  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision (Attack)        : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall / TPR (Attack)     : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score (Attack)         : {f1:.4f}")
print(f"False Positive Rate (FPR) : {fpr:.4f} ({fpr*100:.2f}%)")
print(f"Attack Detection Rate     : {attack_detection_rate:.4f} ({attack_detection_rate*100:.2f}%)")
print(f"ROC-AUC                   : {auc:.4f}")
print("Confusion Matrix:")
print(cm)

print("\nEXACT FOUR-ATTACK-TYPE IDENTIFICATION")
print(f"Overall Exact Attack-Type Accuracy: {exact_accuracy:.4f} ({exact_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(
    actual_attack_types,
    predicted_attack_classes,
    labels=attack_names,
    target_names=attack_names,
    digits=4,
    zero_division=0,
))

print("\nPer-attack results:")
rows = []
for attack_name, expected_label in attack_columns.items():
    hp = df_hybrid[attack_name].values
    rp = df_rules[attack_name].values
    hybrid_exact = int((hp == expected_label).sum())
    rule_exact = int((rp == expected_label).sum())
    hybrid_detected = int(np.array([p in {
        "SHAP-PGD Attack", "DeepFool Attack",
        "Weak Adversarial Attack", "Noise-Based Adversarial Attack"
    } for p in hp]).sum())
    rows.append({
        "Attack Type": attack_name,
        "Hybrid Detection Rate": hybrid_detected / n_eval,
        "Hybrid Exact Identification Rate": hybrid_exact / n_eval,
        "Rule-Based Exact Identification Rate": rule_exact / n_eval,
    })

per_attack_df = pd.DataFrame(rows)
print(per_attack_df.to_string(index=False, float_format=lambda x: f"{x*100:.2f}%"))

cm_exact = confusion_matrix(
    actual_attack_types,
    predicted_attack_classes,
    labels=attack_names,
)
print("\nExact Attack-Type Confusion Matrix:")
print(pd.DataFrame(cm_exact, index=attack_names, columns=attack_names))

print("\nONE-VS-REST TPR / FPR")
metrics = []
for attack in attack_names:
    actual_binary = (actual_attack_types == attack).astype(int)
    predicted_binary = (predicted_attack_classes == attack).astype(int)
    cm1 = confusion_matrix(actual_binary, predicted_binary, labels=[0, 1])
    tn, fp, fn, tp = cm1[0, 0], cm1[0, 1], cm1[1, 0], cm1[1, 1]
    tpr = tp / (tp + fn) if tp + fn > 0 else 0.0
    fpr1 = fp / (fp + tn) if fp + tn > 0 else 0.0
    metrics.append({"Attack Type": attack, "TP": tp, "FN": fn, "FP": fp, "TN": tn, "TPR": tpr, "FPR": fpr1})
metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False, formatters={"TPR": lambda x: f"{x*100:.2f}%", "FPR": lambda x: f"{x*100:.2f}%"}))

# Save combined artifacts for the paper/results section.
combined_file = "/kaggle/working/run2_combined_results_dir"
os.makedirs(combined_dir, exist_ok=True)
df_hybrid.to_csv(os.path.join(combined_dir, "hybrid_predictions_10000.csv"), index=False)
df_rules.to_csv(os.path.join(combined_dir, "rule_predictions_10000.csv"), index=False)
per_attack_df.to_csv(os.path.join(combined_dir, "per_attack_metrics.csv"), index=False)
metrics_df.to_csv(os.path.join(combined_dir, "attack_type_ovr_metrics.csv"), index=False)
np.savetxt(os.path.join(combined_dir, "exact_attack_type_confusion_matrix.csv"), cm_exact, fmt="%d", delimiter=",")

summary = {
    "n_eval_per_attack": int(n_eval),
    "total_attack_instances": int(4*n_eval),
    "dnn_accuracy": float(accuracy),
    "dnn_precision_attack": float(precision),
    "dnn_recall_attack": float(recall),
    "dnn_f1_attack": float(f1),
    "dnn_fpr": float(fpr),
    "dnn_attack_detection_rate": float(attack_detection_rate),
    "dnn_roc_auc": float(auc),
    "hybrid_exact_attack_type_accuracy": float(exact_accuracy),
}
joblib.dump(summary, os.path.join(combined_dir, "final_summary.joblib"))

print("\nFinal files saved to:", combined_dir)
for f in sorted(os.listdir(combined_dir)):
    print(" ", f)

